## Step 1 - Create a Realistic Synthetic Dataset

In [1]:
import numpy as np
import pandas as pd

np.random.seed(42)

n = 2000

time = pd.date_range(start='2024-01-01', periods=n, freq='H')

# Normal conditions
pressure_in = np.random.normal(100, 2, n)
pressure_out = pressure_in - np.random.normal(5, 1, n)

flow_in = np.random.normal(500, 20, n)
flow_out = flow_in - np.random.normal(3, 2, n)

temperature = np.random.normal(30, 1, n)

# Introduce leak events
leak = np.zeros(n)

leak_indices = np.random.choice(range(200, 1800), size=150, replace=False)

for i in leak_indices:
    leak[i] = 1
    pressure_out[i] -= np.random.uniform(5, 10)
    flow_out[i] -= np.random.uniform(10, 30)

df = pd.DataFrame({
    'time': time,
    'pressure_in': pressure_in,
    'pressure_out': pressure_out,
    'flow_in': flow_in,
    'flow_out': flow_out,
    'temperature': temperature,
    'leak': leak
})

df.to_csv("pipeline_data.csv", index=False)

In [4]:
df.tail()

,time,pressure_in,pressure_out,flow_in,flow_out,temperature,leak
1995,2024-03-24 03:00:00,102.140300,97.111843,503.429386,496.009675,31.301102,0.0
1996,2024-03-24 04:00:00,99.946957,97.024769,523.052962,519.398973,28.001655,0.0
1997,2024-03-24 05:00:00,98.236251,93.556549,475.651924,474.322173,29.294683,0.0
1998,2024-03-24 06:00:00,99.673866,93.030488,509.359008,503.053182,30.495766,0.0
1999,2024-03-24 07:00:00,98.510195,93.149547,476.594386,469.438831,30.644388,0.0


## Step 2 - Feature Engineering

In [5]:
df['pressure_diff'] = df['pressure_in'] - df['pressure_out']
df['flow_diff'] = df['flow_in'] - df['flow_out']

# Time-series features
df['pressure_roll_mean'] = df['pressure_diff'].rolling(5).mean()
df['flow_roll_std'] = df['flow_diff'].rolling(5).std()

df = df.dropna()
df

,time,pressure_in,pressure_out,flow_in,flow_out,temperature,leak,pressure_diff,flow_diff,pressure_roll_mean,flow_roll_std
4,2024-01-01 04:00:00,99.531693,96.425308,472.662833,470.091133,31.277857,0.0,3.106385,2.571699,4.237261,0.702472
5,2024-01-01 05:00:00,99.531726,94.318432,511.851345,507.177037,30.570487,0.0,5.213294,4.674309,4.414956,1.372497
6,2024-01-01 06:00:00,103.158426,98.157220,445.912167,443.554485,30.101723,0.0,5.001205,2.357682,4.444101,1.324854
7,2024-01-01 07:00:00,101.534869,97.351958,487.402309,487.573560,31.498012,0.0,4.182911,-0.171251,4.439167,1.730104
8,2024-01-01 08:00:00,99.061051,93.401806,490.234523,484.954387,29.687164,0.0,5.659246,5.280136,4.632608,2.158625
...,...,...,...,...,...,...,...,...,...,...,...
1995,2024-03-24 03:00:00,102.140300,97.111843,503.429386,496.009675,31.301102,0.0,5.028458,7.419711,4.253544,2.378006
1996,2024-03-24 04:00:00,99.946957,97.024769,523.052962,519.398973,28.001655,0.0,2.922188,3.653989,3.932235,2.273981
1997,2024-03-24 05:00:00,98.236251,93.556549,475.651924,474.322173,29.294683,0.0,4.679702,1.329751,4.263273,2.507796
1998,2024-03-24 06:00:00,99.673866,93.030488,509.359008,503.053182,30.495766,0.0,6.643378,6.305826,4.441729,2.742058


## Step 3 - Build the Model

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

features = [
    'pressure_diff',
    'flow_diff',
    'pressure_roll_mean',
    'flow_roll_std'
]

X = df[features]
y = df['leak']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y)

model = RandomForestClassifier()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00       370
         1.0       1.00      1.00      1.00        30

    accuracy                           1.00       400
   macro avg       1.00      1.00      1.00       400
weighted avg       1.00      1.00      1.00       400

ROC-AUC: 1.0


## Step 4 - Deployment

In [7]:
import streamlit as st

st.title("Pipeline Leak Detection")

pressure_diff = st.number_input("Pressure Difference")
flow_diff = st.number_input("Flow Difference")

if st.button("Predict"):
    prediction = model.predict([[pressure_diff, flow_diff, 0, 0]])
    st.write("Leak Detected" if prediction[0] == 1 else "Normal")

2026-04-22 11:32:56.826 
  command:

    streamlit run C:\Users\User\anaconda3\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
